In [ ]:
!pip install -q gradio scikit-learn pandas matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving jd_analyzer_dataset.csv to jd_analyzer_dataset (2).csv


In [ ]:
df = pd.read_csv("jd_analyzer_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

In [ ]:
print("Job Categories:\n")

for category in df["job_category"].unique():
    print("-", category)

In [ ]:
plt.figure(figsize=(10, 5))

df["job_category"].value_counts().plot(kind="bar")

plt.title("Job Category Distribution")
plt.xlabel("Job Category")
plt.ylabel("Number of Job Descriptions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
def clean_text(text):
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


df["clean_description"] = df["job_description"].apply(clean_text)

df[["job_description", "clean_description"]].head()

In [ ]:
X = df["clean_description"]
y = df["job_category"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF conversion completed!")

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model training completed successfully!")

In [ ]:
y_pred = model.predict(X_test_tfidf)

print("Predictions generated successfully!")

print("\nFirst 10 predictions:")
for actual, predicted in zip(y_test.iloc[:10], y_pred[:10]):
    print("Actual:", actual, "| Predicted:", predicted)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy: {:.2f}%".format(accuracy * 100))


In [ ]:
print("Classification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

categories = sorted(df["job_category"].unique())

plt.figure(figsize=(10, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=categories,
    yticklabels=categories
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
sample_jd = """
We are looking for a Data Scientist with experience in Python,
Pandas, NumPy, SQL and Machine Learning.
The candidate should also understand statistics and data visualization.
"""

# Clean the job description
cleaned_text = clean_text(sample_jd)

# Convert text into TF-IDF features
sample_vector = tfidf.transform([cleaned_text])

# Predict job category
prediction = model.predict(sample_vector)[0]

# Get prediction confidence
probability = model.predict_proba(sample_vector).max()

print("Job Description:")
print(sample_jd)

print("\nPredicted Job Category:")
print(prediction)

print("\nPrediction Confidence:")
print("{:.2f}%".format(probability * 100))

In [ ]:
# Create a dictionary from the dataset
category_skills = {}

for category in df["job_category"].unique():

    # Get all rows belonging to this category
    category_rows = df[df["job_category"] == category]

    # Collect all skills from required_skills column
    skills = set()

    for skill_text in category_rows["required_skills"]:

        skill_list = skill_text.split(",")

        for skill in skill_list:
            skills.add(skill.strip())

    category_skills[category] = sorted(skills)


# Display the skills learned from the dataset
for category, skills in category_skills.items():

    print("\n", category)
    print(skills)

In [ ]:
def skill_gap_detector(predicted_category, candidate_skills):

    # Get required skills from dataset
    required_skills = category_skills[predicted_category]

    # Convert candidate input into a list
    candidate_skill_list = [
        skill.strip().lower()
        for skill in candidate_skills.split(",")
    ]

    # Required skills in lowercase
    required_lower = {
        skill.lower(): skill
        for skill in required_skills
    }

    # Find matching skills
    matched_skills = []

    for skill in candidate_skill_list:

        if skill in required_lower:
            matched_skills.append(required_lower[skill])

    # Find missing skills
    missing_skills = []

    for skill in required_skills:

        if skill.lower() not in candidate_skill_list:
            missing_skills.append(skill)

    # Calculate percentage
    if len(required_skills) > 0:

        match_percentage = (
            len(matched_skills) /
            len(required_skills)
        ) * 100

    else:

        match_percentage = 0

    return (
        required_skills,
        matched_skills,
        missing_skills,
        match_percentage
    )

In [ ]:
predicted_category = "Data Science"

candidate_skills = "Python, SQL, HTML, CSS"

required, matched, missing, percentage = skill_gap_detector(
    predicted_category,
    candidate_skills
)

print("Predicted Job Category:")
print(predicted_category)

print("\nRequired Skills:")
print(", ".join(required))

print("\nCandidate Skills:")
print(candidate_skills)

print("\nMatched Skills:")
for skill in matched:
    print("✓", skill)

print("\nMissing Skills:")
for skill in missing:
    print("✗", skill)

print("\nSkill Match Percentage:")
print("{:.2f}%".format(percentage))

In [ ]:
def complete_analyzer(job_description, candidate_skills):

    # -----------------------------------
    # STEP 1: Clean job description
    # -----------------------------------

    cleaned_text = clean_text(job_description)


    # -----------------------------------
    # STEP 2: Convert text to TF-IDF
    # -----------------------------------

    job_vector = tfidf.transform([cleaned_text])


    # -----------------------------------
    # STEP 3: Predict job category
    # -----------------------------------

    predicted_category = model.predict(job_vector)[0]


    # -----------------------------------
    # STEP 4: Prediction confidence
    # -----------------------------------

    confidence = model.predict_proba(job_vector).max()


    # -----------------------------------
    # STEP 5: Get required skills
    # from the DATASET
    # -----------------------------------

    required_skills = category_skills[predicted_category]


    # -----------------------------------
    # STEP 6: Process candidate skills
    # -----------------------------------

    candidate_list = [
        skill.strip().lower()
        for skill in candidate_skills.split(",")
    ]


    # -----------------------------------
    # STEP 7: Find matched skills
    # -----------------------------------

    matched_skills = []

    for skill in required_skills:

        if skill.lower() in candidate_list:

            matched_skills.append(skill)


    # -----------------------------------
    # STEP 8: Find missing skills
    # -----------------------------------

    missing_skills = []

    for skill in required_skills:

        if skill.lower() not in candidate_list:

            missing_skills.append(skill)


    # -----------------------------------
    # STEP 9: Calculate skill match
    # -----------------------------------

    if len(required_skills) > 0:

        match_percentage = (
            len(matched_skills) /
            len(required_skills)
        ) * 100

    else:

        match_percentage = 0


    # -----------------------------------
    # STEP 10: Return results
    # -----------------------------------

    return {
        "category": predicted_category,
        "confidence": confidence * 100,
        "required_skills": required_skills,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "match_percentage": match_percentage
    }

In [ ]:
import gradio as gr


def gradio_analyzer(job_description, candidate_skills):

    if not job_description.strip():

        return (
            "Please enter a job description.",
            "",
            "",
            "",
            ""
        )


    if not candidate_skills.strip():

        return (
            "Please enter your skills.",
            "",
            "",
            "",
            ""
        )


    result = complete_analyzer(
        job_description,
        candidate_skills
    )


    category = result["category"]

    confidence = "{:.2f}%".format(
        result["confidence"]
    )


    matched = "\n".join(
        ["✓ " + skill
         for skill in result["matched_skills"]]
    )


    missing = "\n".join(
        ["✗ " + skill
         for skill in result["missing_skills"]]
    )


    match = "{:.2f}%".format(
        result["match_percentage"]
    )


    return (
        category,
        confidence,
        matched,
        missing,
        match
    )

In [ ]:
with gr.Blocks(
    title="JD Analyzer & Skill Gap Detector"
) as demo:

    gr.Markdown(
        """
        # 📄 JD Analyzer & Skill Gap Detector

        ### Machine Learning Mini Project

        Analyze a job description and identify
        the candidate's skill gaps.
        """
    )


    with gr.Row():

        with gr.Column():

            job_input = gr.Textbox(
                label="Job Description",
                placeholder="Paste the job description here...",
                lines=10
            )


            skills_input = gr.Textbox(
                label="Your Skills",
                placeholder="Example: Python, SQL, HTML, CSS",
                lines=4
            )


            analyze_button = gr.Button(
                "🔍 Analyze"
            )


        with gr.Column():

            category_output = gr.Textbox(
                label="Predicted Job Category"
            )


            confidence_output = gr.Textbox(
                label="Prediction Confidence"
            )


            matched_output = gr.Textbox(
                label="Matched Skills",
                lines=6
            )


            missing_output = gr.Textbox(
                label="Missing Skills",
                lines=8
            )


            match_output = gr.Textbox(
                label="Overall Skill Match"
            )


    analyze_button.click(
        fn=gradio_analyzer,

        inputs=[
            job_input,
            skills_input
        ],

        outputs=[
            category_output,
            confidence_output,
            matched_output,
            missing_output,
            match_output
        ]
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://863d65bf4d086f2b0c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
